# 日本語の文章の読みやすさを測る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_gec_education/readability_ja.ipynb)

NLP クックブック 28-17 ／ 文法誤り訂正と言語教育

日本語の文章の難しさを、文の長さ・漢字の割合・語種などの簡単な指標で数値化したい（教材選び、やさしい日本語の確認）。

In [1]:
!pip install -q fugashi unidic-lite regex

In [2]:
import re
import pandas as pd
import regex
from fugashi import Tagger

tagger = Tagger()

def readability(text):
    sents = [s for s in re.split(r"(?<=[。！？])", text) if s.strip()]
    words = [(w.surface, w.feature.pos1, w.feature.goshu) for s in sents for w in tagger(s) if w.feature.pos1 not in ("補助記号", "空白")]
    chars = regex.findall(r"\S", text)
    content = [w for w in words if w[1] in ("名詞", "動詞", "形容詞", "形状詞", "副詞")]
    return {
        "文数": len(sents),
        "平均文長 (語)": round(len(words) / len(sents), 1),
        "漢字率": round(len(regex.findall(r"\p{Han}", text)) / len(chars), 2),
        "カタカナ率": round(len(regex.findall(r"\p{Katakana}", text)) / len(chars), 2),
        "漢語率": round(sum(w[2] == "漢" for w in content) / len(content), 2),
        "和語率": round(sum(w[2] == "和" for w in content) / len(content), 2),
        "読点/文": round(text.count("、") / len(sents), 1),
    }

texts = {
    "やさしい日本語": "じしんが おきました。たてものの そとに でて ください。エレベーターは つかわないで ください。ちかくの がっこうに にげて ください。",
    "新聞": "政府は 12 日、来年度予算案を閣議決定した。一般会計の歳出総額は過去最大を更新し、社会保障費と防衛費の増加が全体を押し上げた。",
    "法律": "この法律は、個人情報の適正な取扱いに関し、基本理念及び政府による基本方針の作成その他の個人情報の保護に関する施策の基本となる事項を定め、国及び地方公共団体の責務等を明らかにすることを目的とする。",
}
pd.DataFrame({k: readability(v) for k, v in texts.items()})

,やさしい日本語,新聞,法律
文数,4.00,2.00,1.00
平均文長 (語),6.20,18.50,55.00
漢字率,0.00,0.66,0.55
カタカナ率,0.07,0.00,0.00
漢語率,0.15,0.82,0.67
和語率,0.77,0.14,0.27
読点/文,0.00,1.00,3.00


- 文が長く、漢語（音読みの語）が多いほど難しく感じられる。UniDic の語種 (`goshu`) で和語・漢語・外来語を区別できる。
- 学習者向けには、語彙の難易度（日本語能力試験の級別語彙など）や、係り受けの距離も重要な手がかりになる。
- 英語の Flesch-Kincaid（この章の「英文の読みやすさ（リーダビリティ）を測る」）のような定番の式は日本語にはない。日本語の公開ツールとしては jReadability などがある。